# Play Tennis Prediction Using a CART Decision Tree

This is an alternative implementation of the PlayTennis classification example. It uses a CART decision tree with **Gini impurity** to predict whether a match can be played based on weather conditions.

**Workflow**
- Build and inspect the weather dataset
- Explore the target classes
- Compare candidate binary splits using Gini impurity
- Encode categorical columns and divide the data
- Train and evaluate a decision tree
- Visualize the rules and predict unseen weather conditions

## 1. Import libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

pd.set_option("display.max_columns", None)

## 2. Prepare the weather dataset
Each record represents one day. `PlayTennis` is the class we want the model to predict.

In [ ]:
weather_data = {
    "Outlook": ["Sunny", "Sunny", "Overcast", "Rain", "Rain", "Rain", "Overcast",
                "Sunny", "Sunny", "Rain", "Sunny", "Overcast", "Overcast", "Rain"],
    "Temperature": ["Hot", "Hot", "Hot", "Mild", "Cool", "Cool", "Cool",
                    "Mild", "Cool", "Mild", "Mild", "Mild", "Hot", "Mild"],
    "Humidity": ["High", "High", "High", "High", "Normal", "Normal", "Normal",
                 "High", "Normal", "Normal", "Normal", "High", "Normal", "High"],
    "Wind": ["Weak", "Strong", "Weak", "Weak", "Weak", "Strong", "Strong",
             "Weak", "Weak", "Weak", "Strong", "Strong", "Weak", "Strong"],
    "PlayTennis": ["No", "No", "Yes", "Yes", "Yes", "No", "Yes",
                   "No", "Yes", "Yes", "Yes", "Yes", "Yes", "No"]
}

weather_df = pd.DataFrame(weather_data)
display(weather_df)

In [ ]:
print("Dataset shape:", weather_df.shape)
print("\nMissing values:")
print(weather_df.isnull().sum())
print("\nClass frequency:")
print(weather_df["PlayTennis"].value_counts())

## 3. Visualize the class distribution

In [ ]:
class_frequency = weather_df["PlayTennis"].value_counts()
class_frequency.plot(kind="bar", figsize=(6, 4), rot=0)
plt.title("Number of Play and No-Play Days")
plt.xlabel("Play tennis?")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

## 4. Calculate Gini impurity and compare splits

CART creates binary splits. Gini impurity measures how mixed the target classes are in a group:

\[Gini = 1 - \sum p_i^2\]

A lower weighted child Gini indicates a better split.

In [ ]:
target = "PlayTennis"
input_features = ["Outlook", "Temperature", "Humidity", "Wind"]

def calculate_gini(frame):
    if len(frame) == 0:
        return 0.0
    proportions = frame[target].value_counts(normalize=True)
    return 1 - (proportions ** 2).sum()

root_gini = calculate_gini(weather_df)
print(f"Gini impurity before splitting: {root_gini:.4f}")

In [ ]:
candidate_splits = []

for column in input_features:
    for category in weather_df[column].unique():
        group_a = weather_df[weather_df[column] == category]
        group_b = weather_df[weather_df[column] != category]
        weighted_gini = (len(group_a) / len(weather_df)) * calculate_gini(group_a) + \
                        (len(group_b) / len(weather_df)) * calculate_gini(group_b)
        candidate_splits.append({
            "Feature": column,
            "Rule": f"{column} = {category}",
            "Weighted Gini": weighted_gini,
            "Gini Gain": root_gini - weighted_gini
        })

split_table = pd.DataFrame(candidate_splits).sort_values("Weighted Gini").reset_index(drop=True)
display(split_table.round(4))
print("Best candidate:", split_table.loc[0, "Rule"])

## 5. Split and encode the features
We split the original rows before fitting the encoder. This prevents information from the test set from influencing preprocessing.

In [ ]:
features = weather_df[input_features]
labels = weather_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.25, random_state=21, stratify=labels
)

encoder = ColumnTransformer(
    [("weather_categories", OneHotEncoder(handle_unknown="ignore"), input_features)],
    remainder="drop"
)

X_train_ready = encoder.fit_transform(X_train)
X_test_ready = encoder.transform(X_test)
feature_labels = encoder.get_feature_names_out()

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])
display(pd.DataFrame(X_train_ready.toarray() if hasattr(X_train_ready, "toarray") else X_train_ready,
                     columns=feature_labels).head())

## 6. Fit the CART model
The `gini` criterion makes the classifier use Gini impurity. A limited depth keeps the tree readable for this small example.

In [ ]:
tennis_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    min_samples_leaf=1,
    random_state=21
)

tennis_tree.fit(X_train_ready, y_train)
print("Decision tree training completed.")

## 7. Evaluate predictions

In [ ]:
test_predictions = tennis_tree.predict(X_test_ready)
model_accuracy = accuracy_score(y_test, test_predictions)

print(f"Accuracy on test data: {model_accuracy:.2%}")
print("\nDetailed classification report:")
print(classification_report(y_test, test_predictions, zero_division=0))

In [ ]:
matrix = confusion_matrix(y_test, test_predictions, labels=["No", "Yes"])
print("Confusion matrix:\n", matrix)
ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=["No", "Yes"]).plot(cmap="Purples")
plt.title("PlayTennis Prediction Results")
plt.tight_layout()
plt.show()

## 8. Display the decision tree

In [ ]:
plt.figure(figsize=(17, 8))
plot_tree(
    tennis_tree,
    feature_names=feature_labels,
    class_names=tennis_tree.classes_,
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("CART Rules for PlayTennis")
plt.tight_layout()
plt.show()

## 9. Predict for new weather conditions

In [ ]:
upcoming_weather = pd.DataFrame({
    "Outlook": ["Sunny", "Overcast", "Rain"],
    "Temperature": ["Cool", "Mild", "Mild"],
    "Humidity": ["High", "High", "Normal"],
    "Wind": ["Strong", "Weak", "Strong"]
})

upcoming_encoded = encoder.transform(upcoming_weather)
upcoming_weather["Prediction"] = tennis_tree.predict(upcoming_encoded)
upcoming_weather["Confidence"] = tennis_tree.predict_proba(upcoming_encoded).max(axis=1).round(3)
display(upcoming_weather)

## 10. Summary

- CART builds a tree by repeatedly dividing records into two groups.
- Gini impurity measures how mixed the target classes are.
- One-hot encoding converts weather categories into numeric input columns.
- The confusion matrix and classification report help assess predictions.
- Since this dataset contains only 14 records, the evaluation score is illustrative and may change with a different split.